# Light Attenuation (KD490) — All Norway

Downloads KD490 (diffuse attenuation at 490 nm) from CMEMS for 2025,
covering all of Norway using two products:
- **Atlantic** (1 km): `cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D`
- **Arctic** (4 km): `cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D`

The bounding box is derived from the Norway DEM raster. Atlantic is preferred in the overlap zone.
Uses GDAL for all raster operations to keep memory usage low.
The photic zone is computed at DEM resolution (50 m).

In [ ]:
from pathlib import Path

import copernicusmarine as cm
import numpy as np
import xarray as xr
from osgeo import gdal, osr
from osgeo_utils import gdal_calc
from rasterio.warp import transform_bounds

gdal.UseExceptions()

out_dir = Path("../niva")
out_dir.mkdir(exist_ok=True)

GTIFF_CO = ["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=256", "BLOCKYSIZE=256", "BIGTIFF=IF_SAFER"]

kd_year = "2024"
"""Year to extract kd490 values from cmems"""

'Year to extract kd490 values from cmems'

## 1. Bounding box from Norway DEM

In [2]:
dem_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"

ds = gdal.Open(dem_url)
gt = ds.GetGeoTransform()
dem_xsize, dem_ysize = ds.RasterXSize, ds.RasterYSize
dem_left = gt[0]
dem_top = gt[3]
dem_right = dem_left + gt[1] * dem_xsize
dem_bottom = dem_top + gt[5] * dem_ysize
dem_srs = ds.GetSpatialRef()
ds = None

lon_min, lat_min, lon_max, lat_max = transform_bounds(
    "EPSG:25833", "EPSG:4326", dem_left, dem_bottom, dem_right, dem_top,
)
lon_min, lat_min = round(lon_min - 0.5, 1), round(lat_min - 0.5, 1)
lon_max, lat_max = round(lon_max + 0.5, 1), round(lat_max + 0.5, 1)

print(f"Norway bbox (WGS84): lon [{lon_min}, {lon_max}], lat [{lat_min}, {lat_max}]")
print(f"DEM grid: {dem_xsize} x {dem_ysize} @ 50 m, bounds: [{dem_left}, {dem_bottom}, {dem_right}, {dem_top}]")

Norway bbox (WGS84): lon [-2.2, 32.8], lat [57.0, 72.3]
DEM grid: 24431 x 30735 @ 50 m, bounds: [-99600.0, 6426000.0, 1121950.0, 7962750.0]


## 2. Download KD490 from CMEMS (2025)

In [3]:
start_date = f"{kd_year}-04-01"
end_date = f"{kd_year}-10-31"
print(f"Time range: {start_date} to {end_date}")

OVERLAP_LAT = 62.0

ds_atl = cm.open_dataset(
    dataset_id="cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=lat_min,
    maximum_latitude=min(lat_max, 66.0),
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Atlantic shape: {ds_atl['KD490'].shape}")

Time range: 2025-04-01 to 2025-10-31


INFO - 2026-08-18T12:08:46Z - Selected dataset version: "202603"
INFO - 2026-08-18T12:08:46Z - Selected dataset part: "default"
WARNING - 2026-08-18T12:08:46Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-18T12:08:46Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]
WARNING - 2026-08-18T12:08:46Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-18T12:08:46Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]


Atlantic shape: (214, 864, 1459)


In [4]:
ds_arc = cm.open_dataset(
    dataset_id="cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=OVERLAP_LAT,
    maximum_latitude=lat_max,
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Arctic shape: {ds_arc['KD490'].shape}")

INFO - 2026-08-18T12:08:48Z - Selected dataset version: "202311"
INFO - 2026-08-18T12:08:48Z - Selected dataset part: "default"
WARNING - 2026-08-18T12:08:48Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]
WARNING - 2026-08-18T12:08:48Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]


Arctic shape: (214, 210, 389)


## 3. Compute temporal mean and mosaic

Prefer Atlantic (1 km) in overlap zone; fill gaps with Arctic (4 km interpolated to same grid).

In [5]:
kd_atl_mean = ds_atl["KD490"].mean(dim="time", skipna=True).compute()
kd_arc_mean = ds_arc["KD490"].mean(dim="time", skipna=True).compute()

# Interpolate both onto a common 0.01° grid
res_deg = 0.01
lons = np.arange(lon_min, lon_max, res_deg)
lats = np.arange(lat_max, lat_min, -res_deg)

kd_atl_interp = kd_atl_mean.interp(
    latitude=xr.DataArray(lats, dims="y"),
    longitude=xr.DataArray(lons, dims="x"),
    method="nearest",
).values.astype(np.float32)

kd_arc_interp = kd_arc_mean.interp(
    latitude=xr.DataArray(lats, dims="y"),
    longitude=xr.DataArray(lons, dims="x"),
    method="nearest",
).values.astype(np.float32)

kd_merged = np.where(np.isfinite(kd_atl_interp), kd_atl_interp, kd_arc_interp)

print(f"Atlantic valid: {np.sum(np.isfinite(kd_atl_interp))}")
print(f"Arctic valid:   {np.sum(np.isfinite(kd_arc_interp))}")
print(f"Merged valid:   {np.sum(np.isfinite(kd_merged))}")

# Free xarray datasets
del ds_atl, ds_arc, kd_atl_mean, kd_arc_mean, kd_atl_interp, kd_arc_interp

Atlantic valid: 932824
Arctic valid:   1396504
Merged valid:   2329328


## 4. Save merged KD490 as GeoTIFF (WGS84)

In [6]:
kd_wgs84_path = str(out_dir / f"KD490_norge_{kd_year}_wgs84.tif")

drv = gdal.GetDriverByName("GTiff")
ny, nx = kd_merged.shape
ds_out = drv.Create(kd_wgs84_path, nx, ny, 1, gdal.GDT_Float32, options=GTIFF_CO)
ds_out.SetGeoTransform([lon_min, res_deg, 0, lat_max, 0, -res_deg])
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)
ds_out.SetProjection(srs.ExportToWkt())
band = ds_out.GetRasterBand(1)
band.SetNoDataValue(float("nan"))
band.WriteArray(kd_merged)
ds_out.FlushCache()
ds_out = None
del kd_merged

print(f"Saved WGS84 mosaic: {kd_wgs84_path}")

Saved WGS84 mosaic: ../niva/KD490_norge_2025_wgs84.tif


## 5. Reproject to EPSG:25833 at 1000 m and save (before fill)

In [7]:
kd_25833_path = str(out_dir / f"KD490_norge_{kd_year}_25833.tif")

gdal.Warp(
    kd_25833_path, kd_wgs84_path,
    dstSRS="EPSG:25833",
    xRes=1000, yRes=1000,
    outputBounds=[dem_left, dem_bottom, dem_right, dem_top],
    resampleAlg=gdal.GRA_Bilinear,
    srcNodata=float("nan"), dstNodata=float("nan"),
    creationOptions=GTIFF_CO,
)

ds = gdal.Open(kd_25833_path)
data = ds.GetRasterBand(1).ReadAsArray()
print(f"KD490 25833 — shape: {data.shape}, valid: {np.sum(np.isfinite(data))}/{data.size}")
ds = None
del data

print(f"Saved (unfilled): {kd_25833_path}")

KD490 25833 — shape: (1537, 1222), valid: 647994/1878214
Saved (unfilled): ../niva/KD490_norge_2025_25833.tif


## 6. Gap-fill KD490 using GDAL FillNodata

In [8]:
kd_filled_path = str(out_dir / f"KD490_norge_{kd_year}_filled_25833.tif")

# Copy to filled output
gdal.GetDriverByName("GTiff").CopyFiles(kd_filled_path, kd_25833_path)

# Fine fill
print("Fine fill (maxSearchDist=200)...")
ds = gdal.Open(kd_filled_path, gdal.GA_Update)
band = ds.GetRasterBand(1)
gdal.FillNodata(
    band,
    maskBand=band.GetMaskBand(),
    maxSearchDist=200, smoothingIterations=0,
    callback=gdal.TermProgress_nocb,
)
ds.FlushCache()
ds = None

# Coarse fill: downsample, fill large gaps, upsample, merge
print("Coarse fill (20x downsample, maxSearchDist=500)...")
tmp_coarse = str(out_dir / "tmp_kd490_coarse.tif")
tmp_coarse_up = str(out_dir / "tmp_kd490_coarse_up.tif")

gdal.Warp(
    tmp_coarse, kd_filled_path,
    xRes=20000, yRes=20000,
    resampleAlg=gdal.GRA_Average,
    srcNodata=float("nan"), dstNodata=float("nan"),
    creationOptions=["COMPRESS=DEFLATE"],
)

ds_c = gdal.Open(tmp_coarse, gdal.GA_Update)
gdal.FillNodata(
    ds_c.GetRasterBand(1),
    maskBand=ds_c.GetRasterBand(1).GetMaskBand(),
    maxSearchDist=500, smoothingIterations=0,
    callback=gdal.TermProgress_nocb,
)
ds_c.FlushCache()
ds_c = None

# Upsample coarse back to 1 km
ds_ref = gdal.Open(kd_filled_path)
gdal.Warp(
    tmp_coarse_up, tmp_coarse,
    width=ds_ref.RasterXSize, height=ds_ref.RasterYSize,
    outputBounds=[dem_left, dem_bottom, dem_right, dem_top],
    resampleAlg=gdal.GRA_Bilinear,
    srcNodata=float("nan"), dstNodata=float("nan"),
    creationOptions=GTIFF_CO,
)
ds_ref = None

# Merge: fine-filled where available, coarse elsewhere
gdal_calc.Calc(
    calc="numpy.where(numpy.isnan(A), B, A)",
    outfile=kd_filled_path,
    A=kd_filled_path, B=tmp_coarse_up,
    type="Float32", NoDataValue=float("nan"), hideNoData=True,
    creation_options=GTIFF_CO,
    overwrite=True,
)

# Cleanup
for p in [tmp_coarse, tmp_coarse_up]:
    Path(p).unlink(missing_ok=True)

ds = gdal.Open(kd_filled_path)
data = ds.GetRasterBand(1).ReadAsArray()
print(f"Filled — valid: {np.sum(np.isfinite(data))}/{data.size}")
ds = None
del data
print(f"Saved: {kd_filled_path}")

Fine fill (maxSearchDist=200)...
0...10...20...30...40...50...60...70...80...90...Coarse fill (20x downsample, maxSearchDist=500)...
100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90..Filled — valid: 1875830/1878214
Saved: ../niva/KD490_norge_2025_filled_25833.tif


## 7. Compute photic zone at DEM resolution (50 m)

Resample filled KD490 (1 km) to the DEM grid (50 m) using GDAL, then apply:

$$z_{photic} = \frac{\ln(100)}{K_{d490}} \approx \frac{4.605}{K_{d490}}$$

A pixel is **photic** (1) if `|depth| < z_photic`, **aphotic** (0) otherwise.
Processed block-by-block to avoid loading the full DEM into memory.

In [ ]:
# Resample filled KD490 to DEM grid (50 m)
kd_dem_res_path = str(out_dir / "tmp_kd490_dem_res.tif")

gdal.Warp(
    kd_dem_res_path, kd_filled_path,
    dstSRS="EPSG:25833",
    width=dem_xsize, height=dem_ysize,
    outputBounds=[dem_left, dem_bottom, dem_right, dem_top],
    resampleAlg=gdal.GRA_Bilinear,
    srcNodata=float("nan"), dstNodata=float("nan"),
    creationOptions=GTIFF_CO,
)
print(f"KD490 resampled to DEM grid: {dem_xsize} x {dem_ysize}")

In [ ]:
photic_path = str(out_dir / f"nisjedata-fotisk-sone-kd{kd_year}_norge_2026_25833.tif")

ds_dem = gdal.Open(dem_url)
ds_kd = gdal.Open(kd_dem_res_path)

dem_nodata = ds_dem.GetRasterBand(1).GetNoDataValue()

drv = gdal.GetDriverByName("GTiff")
ds_out = drv.Create(photic_path, dem_xsize, dem_ysize, 1, gdal.GDT_Int16, options=GTIFF_CO)
ds_out.SetGeoTransform(ds_dem.GetGeoTransform())
ds_out.SetProjection(ds_dem.GetProjection())
band_out = ds_out.GetRasterBand(1)
band_out.SetNoDataValue(-1)

band_dem = ds_dem.GetRasterBand(1)
band_kd = ds_kd.GetRasterBand(1)
block_w, block_h = 256, 256
n_photic = 0
n_aphotic = 0

for y_off in range(0, dem_ysize, block_h):
    rows = min(block_h, dem_ysize - y_off)
    for x_off in range(0, dem_xsize, block_w):
        cols = min(block_w, dem_xsize - x_off)

        depth = band_dem.ReadAsArray(x_off, y_off, cols, rows).astype(np.float32)
        kd = band_kd.ReadAsArray(x_off, y_off, cols, rows).astype(np.float32)

        depth_abs = np.abs(depth)
        valid = (
            np.isfinite(depth) & np.isfinite(kd)
            & (kd > 0) & (depth_abs > 0)
        )
        if dem_nodata is not None:
            valid &= depth != dem_nodata

        block = np.full((rows, cols), -1, dtype=np.int16)
        if valid.any():
            photic_depth = np.log(100) / kd[valid]
            is_photic = depth_abs[valid] < photic_depth
            block[valid] = np.where(is_photic, 1, 0).astype(np.int16)
            n_photic += int(is_photic.sum())
            n_aphotic += int((~is_photic).sum())

        band_out.WriteArray(block, x_off, y_off)

band_out.FlushCache()
band_out = None
ds_out.FlushCache()
ds_out = None
ds_dem = None
ds_kd = None

# Cleanup temp resampled KD490
Path(kd_dem_res_path).unlink(missing_ok=True)

print(f"Photic  pixels: {n_photic:,}")
print(f"Aphotic pixels: {n_aphotic:,}")
print(f"Saved: {photic_path}")


KeyboardInterrupt: 